In [3]:
# Scenario: AI Symptom Tracker (Question-Based)
# This system takes a patient's symptom, generates multiple observations using Groq,
# analyzes them, and provides a safe, non-medical recommendation.

from langgraph.graph import StateGraph, END
from typing import TypedDict, List
import requests
from google.colab import userdata

# 1. Define State
class HealthState(TypedDict):
    symptom: str
    observations: List[str]
    analysis: str
    recommendation: str
    steps_done: int


# 2. Define Nodes

# 🔹 Node 1: Generate Observations using Groq
def generate_observation(state: HealthState):
    symptom = state["symptom"]
    groq_api_key = userdata.get("groq_api_key")

    if not groq_api_key:
        raise ValueError("Groq API key not found. Set 'groq_api_key'.")

    # 🔥 Prompt (LLM instruction)
    prompt = f"""
You are a health assistant (non-diagnostic).

Given the symptom: "{symptom}"

Generate ONE possible general observation such as:
- possible common causes
- related symptoms
- lifestyle factors

IMPORTANT:
- Do NOT give medical diagnosis
- Keep it general and safe
- One observation only (1-2 lines)
"""

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.5
        }
    )

    data = response.json()
    observation = data["choices"][0]["message"]["content"].strip()

    return {
        "observations": state["observations"] + [observation],
        "steps_done": state["steps_done"] + 1
    }


# 🔹 Node 2: Analyze Observations
def analyze_observations(state: HealthState):
    observations = "\n".join(state["observations"])

    groq_api_key = userdata.get("groq_api_key")

    prompt = f"""
You are a health assistant.

Given the following observations:
{observations}

Summarize key insights in 2-3 lines.
Do NOT provide diagnosis.
"""

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    data = response.json()
    analysis = data["choices"][0]["message"]["content"]

    return {"analysis": analysis}


# 🔹 Node 3: Recommendation
def generate_recommendation(state: HealthState):
    analysis = state["analysis"]

    groq_api_key = userdata.get("groq_api_key")

    prompt = f"""
Based on this analysis:
{analysis}

Give a SAFE, non-medical recommendation.

Examples:
- Monitor symptoms
- Stay hydrated
- Consult a doctor if symptoms persist

Do NOT diagnose.
Keep it simple and clear.
"""

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    data = response.json()
    recommendation = data["choices"][0]["message"]["content"]

    return {"recommendation": recommendation}


# 🔹 Conditional Logic (Core Intelligence 🔥)
def check_observations(state: HealthState):
    if len(state["observations"]) < 3:
        return "generate_observation"   # loop
    else:
        return "analyze_observations"   # move ahead


# 3. Build Graph
graph = StateGraph(HealthState)

graph.add_node("generate_observation", generate_observation)
graph.add_node("analyze_observations", analyze_observations)
graph.add_node("generate_recommendation", generate_recommendation)

# Entry
graph.set_entry_point("generate_observation")

# Conditional loop
graph.add_conditional_edges(
    "generate_observation",
    check_observations
)

# Forward flow
graph.add_edge("analyze_observations", "generate_recommendation")
graph.add_edge("generate_recommendation", END)

# Compile
app = graph.compile()


# 4. Run
if __name__ == "__main__":

    # 🔥 USER INPUT (actual prompt)
    symptom_input = input("Enter your symptom: ")

    state = {
        "symptom": symptom_input,
        "observations": [],
        "analysis": "",
        "recommendation": "",
        "steps_done": 0
    }

    result = app.invoke(state)

    print("\n--- Observations ---")
    for obs in result["observations"]:
        print("-", obs)

    print("\n--- Analysis ---")
    print(result["analysis"])

    print("\n--- Recommendation ---")
    print(result["recommendation"])

Enter your symptom: nausea

--- Observations ---
- Possible common causes of nausea include food poisoning, motion sickness, or stomach flu, often triggered by consuming spoiled or fatty foods, or experiencing sudden changes in environment or body position.
- One possible general observation for nausea is that it can be triggered by food poisoning, motion sickness, or stomach upset, often accompanied by symptoms such as vomiting, dizziness, or stomach cramps.
- Possible common causes of nausea include food poisoning, motion sickness, or stress, which can be triggered by a variety of factors such as consuming spoiled food, traveling by car or boat, or experiencing high levels of anxiety.

--- Analysis ---
Key insights suggest that nausea is often triggered by factors such as consuming spoiled or fatty foods, sudden changes in environment or body position, and experiences like motion sickness or food poisoning. Additionally, stress and anxiety can also be contributing factors.

--- Recom

In [4]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List
import requests
from google.colab import userdata

# 1. Define State
class HealthState(TypedDict):
    symptom: str
    observations: List[str]
    analysis: str
    recommendation: str
    steps_done: int


# 🔹 Node 1: Observation Generator (Loop body)
def generate_observation(state: HealthState):
    symptom = state["symptom"]
    groq_api_key = userdata.get("groq_api_key")

    prompt = f"""
You are a safe health assistant.

Given symptom: {symptom}

Generate ONE short general observation (not diagnosis).
"""

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    obs = response.json()["choices"][0]["message"]["content"].strip()

    return {
        "observations": state["observations"] + [obs],
        "steps_done": state["steps_done"] + 1
    }


# 🔹 Loop Controller (THIS IS YOUR WHILE CONDITION 🔥)
def loop_control(state: HealthState):
    if state["steps_done"] < 3:
        return "generate_observation"   # 🔁 loop continue
    else:
        return "analyze"                # 🛑 break loop


# 🔹 Node 2: Analysis
def analyze(state: HealthState):
    obs_text = "\n".join(state["observations"])
    groq_api_key = userdata.get("groq_api_key")

    prompt = f"""
Summarize these observations safely:

{obs_text}

No diagnosis. Just general insight.
"""

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    analysis = response.json()["choices"][0]["message"]["content"]

    return {"analysis": analysis}


# 🔹 Node 3: Recommendation
def recommend(state: HealthState):
    groq_api_key = userdata.get("groq_api_key")

    prompt = f"""
Based on this:
{state['analysis']}

Give safe general recommendation.
No diagnosis.
"""

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    rec = response.json()["choices"][0]["message"]["content"]

    return {"recommendation": rec}


# 3. Build Graph
graph = StateGraph(HealthState)

graph.add_node("generate_observation", generate_observation)
graph.add_node("analyze", analyze)
graph.add_node("recommend", recommend)

# Entry
graph.set_entry_point("generate_observation")

# 🔥 LOOP EDGE (core logic)
graph.add_conditional_edges(
    "generate_observation",
    loop_control
)

# Forward
graph.add_edge("analyze", "recommend")
graph.add_edge("recommend", END)

# Compile
app = graph.compile()


# 4. Run
if __name__ == "__main__":
    symptom_input = input("Enter your symptom: ")

    state = {
        "symptom": symptom_input,
        "observations": [],
        "analysis": "",
        "recommendation": "",
        "steps_done": 0
    }

    result = app.invoke(state)

    print("\nObservations:")
    for o in result["observations"]:
        print("-", o)

    print("\nAnalysis:")
    print(result["analysis"])

    print("\nRecommendation:")
    print(result["recommendation"])

Enter your symptom: headache

Observations:
- Observation: The individual is experiencing discomfort and pain in the head region, which could be related to various underlying factors, such as tension, fatigue, or other potential causes.
- Given the symptom of a headache, here's a short general observation: 

A headache may be a sign of muscle tension, fatigue, or a response to environmental factors such as stress, changes in weather, or loud noise.
- Headaches can often be caused by a combination of factors such as stress, fatigue, or dehydration, making it essential to assess the individual's lifestyle, hydration level, and daily habits.

Analysis:
Here's a summary of the observations safely and generally:

The individual is experiencing discomfort and pain in the head region, which could be related to various underlying factors, such as tension, fatigue, stress, changes in environment, dehydration, or other potential causes. 

Further insight suggests that headaches often result from